In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [2]:
import unicodedata

def normalizar(texto):
    unaccent = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return unaccent.lower()

In [23]:
def buscar_estado(cursor, nome):
    command = """
    SELECT id_estado FROM estado
    WHERE nome = %s
    """
    cursor.execute(command, (nome,))
    id_estado = cursor.fetchone()
    return id_estado[0] if id_estado else None

In [18]:
def inserir_cidade(cursor, nome_cidade, id_estado):
    nome_normalizado = normalizar(nome_cidade)
    comando = """
    INSERT INTO cidade (nome, nome_normalizado, id_estado)
    VALUES (%s, %s, %s)
    ON CONFLICT (nome_normalizado, id_estado) DO NOTHING;
    """
    cursor.execute(comando, (nome_cidade, nome_normalizado, id_estado))

In [ ]:
df = pd.read_csv('../datasets/Cidades.csv', sep=';')
df = df.rename(columns={
    'Nome_UF': 'estado',
    'Nome_Município': 'cidade',
})
df = df[['estado', 'cidade']]

for _, row in df.iterrows():
    estado = row['estado']
    cidade = row['cidade']
    id_estado = buscar_estado(cursor, estado)
    if id_estado is not None:
        inserir_cidade(cursor, cidade, id_estado)

conn.commit()

In [ ]:
conn.rollback()

In [27]:
cursor.close()
conn.close()